# dano_v2 sin Drive
Mismo reentrenamiento que la sección 12 de `eggs_v5.ipynb`, con el dataset tomado de la rama temporal `v5-tmp`. No hay pesos de `dano_v1` en formato Keras a mano, así que parte de ImageNet (80 épocas) con los recortes originales + los de v5.

In [ ]:
%pip install -q ultralytics ai-edge-litert
import json, os, shutil, subprocess, zipfile
from pathlib import Path
import torch
print('GPU:', torch.cuda.is_available())

REPO, TMP = Path('/content/eggs-detector'), Path('/content/tmp')
if not REPO.exists():
    !git clone -q --depth 1 https://github.com/dportilla219/eggs-detector {REPO}
if not TMP.exists():
    !git clone -q --depth 1 -b v5-tmp https://github.com/dportilla219/eggs-detector {TMP}
DATA_DIR = Path('/content/eggs_v2')
for f in sorted(TMP.glob('*.parte1de*')):   # une los trozos de los zips
    nombre, total = f.name.rsplit('.parte1de', 1)
    with open(TMP / nombre, 'wb') as out:
        for i in range(1, int(total) + 1):
            out.write((TMP / f'{nombre}.parte{i}de{total}').read_bytes())
if not (DATA_DIR / 'train').exists():
    for z in sorted(TMP.glob('eggs_v2_parte*.zip')):
        with zipfile.ZipFile(z) as zf:
            zf.extractall(DATA_DIR)
        print('OK', z.name)

PROFE_SRC = Path('/content/huevos_profe')
if not PROFE_SRC.exists():
    !git clone -q https://github.com/adiacla/huevos {PROFE_SRC}
    !git -C {PROFE_SRC} checkout -q 1b4e76826a6c1abfe0a4f20bb16f08b108569012
PUB_SRC = Path('/content/publico_src')
if not PUB_SRC.exists():
    !git clone -q --depth 1 https://github.com/dakshkathuria346-gif/Egg-Defect-Detection {PUB_SRC}

RUN = 'dano_v2'
WORK, SEG_DIR, RUNS_DIR, OUT = Path('/content/dano_work'), Path('/content/seg'), Path('/content/runs'), Path('/content/exports/dano_v2')
WORK.mkdir(exist_ok=True); RUNS_DIR.mkdir(exist_ok=True)
if not (WORK / 'data').exists():
    (WORK / 'data').symlink_to(DATA_DIR)
for f in (REPO / 'dano' / 'anotaciones').glob('*.txt'):
    shutil.copy(f, WORK)
os.chdir(WORK)
os.environ.update({'SEG_DIR': str(SEG_DIR), 'RUNS_DIR': str(RUNS_DIR), 'PYTHONPATH': str(REPO / 'dano'), 'TF_CPP_MIN_LOG_LEVEL': '2'})

D = REPO / 'dano'
if not (SEG_DIR / 'meta.json').exists():
    for split, lst in [('train', 'items_train.txt'), ('train', 'items_neg.txt'), ('valid', 'items_valid.txt'), ('valid', 'items_neg_valid.txt'),
                       ('test', 'items_test.txt'), ('test', 'items_neg_test.txt')]:
        !python {D}/eggmask.py {split} 999999 {lst} 2>&1 | grep -v Warn | tail -1
    SEG_DIR.mkdir(parents=True, exist_ok=True)
    !python {D}/build.py
for s in ['train', 'valid', 'test']:
    fs = list((SEG_DIR / s).glob('*.npz'))
    print(s, len(fs), 'recortes |', sum(f.name.endswith('_p.npz') for f in fs), 'con daño')

reales = []
for e in json.load(open(REPO / 'resultados/v4/etiquetas.json')):
    if e['archivo'][:1].isdigit():
        continue
    c = 1 if e['clase'] == 'Intact' else 0
    f = PUB_SRC / ('non defective' if c == 1 else 'defective') / e['archivo']
    if f.exists():
        reales.append({'file': str(f), 'clase': c, 'cajas': e['cajas'], 'split': e['split']})
print('fotos reales con caja:', len(reales))
json.dump(reales, open('/content/reales.json', 'w'))
V5SEG = Path('/content/v5seg')
if not (V5SEG / 'seg/valid/datos.npz').exists():
    !python {REPO}/v5/datos.py --profe {PROFE_SRC}/imagenes --reales /content/reales.json --dano {REPO}/modelo/eggs_dano_v1_fp16.tflite --out {V5SEG} --n 0 0 0 --seg 3000 300

if not (RUNS_DIR / RUN / 'last.keras').exists():
    shutil.rmtree(RUNS_DIR / RUN, ignore_errors=True)
    !python {REPO}/dano/train_seg_v2.py {RUN} 80 {V5SEG}/seg ninguno 2>&1 | grep -vE "oneDNN|UserWarning|data_adapter|cuda_|computation_placer|absl|E0000|W0000|I0000"
shutil.rmtree(OUT, ignore_errors=True)
!python {REPO}/dano/export_eval.py {RUN} {OUT} {REPO}/modelo/eggs_v2_fp32.tflite 2>&1 | grep -vE "oneDNN|WARNING|absl|E0000|W0000|I0000|cuda_"

paquete = Path('/content/eggs_dano_v2_resultados')
shutil.rmtree(paquete, ignore_errors=True)
shutil.copytree(OUT, paquete) if OUT.exists() else paquete.mkdir()
for f in ['hist.json', 'best.keras', 'last.keras']:
    if (RUNS_DIR / RUN / f).exists():
        shutil.copy2(RUNS_DIR / RUN / f, paquete / f)
z = shutil.make_archive(str(paquete), 'zip', paquete)
from google.colab import files
files.download(z)